# HAHA 2026 — Ensemble (Embeddings + LLM Ollama) — Fase TEST

Notebook para **inferencia sobre el conjunto de test** de HAHA 2026.

Diferencias respecto al notebook de desarrollo:

1. **Entrada por defecto**: archivos `task{1,2,3}_test.tsv` dentro de `haha2026_dev/`.
2. **Fallback**: si los archivos de test aún no están publicados, puedes apuntar a `*_dev.tsv` cambiando `PHASE = "dev"`.
3. **Pool de entrenamiento ampliado**: para los clasificadores (T1, T2) se concatenan `trial + dev` cuando dev tiene etiquetas; si no, solo trial.
4. **Pool RAG ampliado**: para Task 3 se usa todo `task2_trial.tsv` y opcionalmente `task2_dev.tsv` si trae chistes con etiqueta `human`.
5. **Sin métricas**: el test no tiene gold labels; el notebook solo genera las predicciones y empaqueta el zip de submission.

**Estrategia ensemble**

| Tarea | Componente A (embeddings) | Componente B (LLM) | Combinación |
|-------|---------------------------|--------------------|-------------|
| 1 | KNN/LogReg sobre BGE-M3 | Ollama zero-shot | Promedio ponderado |
| 2 | KNN/LogReg sobre BGE-M3 | Ollama zero-shot | Promedio ponderado |
| 3 | RAG (retrieval del trial humano) | Ollama generation few-shot | Generación condicionada |

## 1. Instalación de dependencias

In [ ]:
!pip install -q sentence-transformers pandas scikit-learn numpy ollama tqdm

## 2. Configuración

Cambia `PHASE` según la fase de evaluación:
- `PHASE = "test"`  → lee `task{1,2,3}_test.tsv`
- `PHASE = "dev"`   → lee `task{1,2,3}_dev.tsv` (útil para validar antes del test)

Para Ollama Cloud, define las variables de entorno `OLLAMA_MODE=cloud` y `OLLAMA_API_KEY=...`.

In [ ]:
import os
from pathlib import Path

# ─────────────────────────────────────────
# FASE
# ─────────────────────────────────────────
PHASE = os.environ.get("HAHA_PHASE", "test")  # "test" | "dev"

# ─────────────────────────────────────────
# DATOS
# ─────────────────────────────────────────
DATA_DIR = "haha2026_test"

# Trial (siempre etiquetado, la fuente de entrenamiento principal)
TRIAL_FILE_1 = f"{DATA_DIR}/task1_trial.tsv"
TRIAL_FILE_2 = f"{DATA_DIR}/task2_trial.tsv"

# Dev (en HAHA 2026 no trae labels; lo usamos solo si añade columna tag)
DEV_FILE_1 = f"{DATA_DIR}/task1_dev.tsv"
DEV_FILE_2 = f"{DATA_DIR}/task2_dev.tsv"
DEV_FILE_3 = f"{DATA_DIR}/task3_dev.tsv"

# Eval files: cambian según la fase
if PHASE == "test":
    EVAL_FILE_1 = f"{DATA_DIR}/task1_test.tsv"
    EVAL_FILE_2 = f"{DATA_DIR}/task2_test.tsv"
    EVAL_FILE_3 = f"{DATA_DIR}/task3_test.tsv"
else:
    EVAL_FILE_1 = DEV_FILE_1
    EVAL_FILE_2 = DEV_FILE_2
    EVAL_FILE_3 = DEV_FILE_3

# Salida
OUT_FILE_1 = "task1.tsv"
OUT_FILE_2 = "task2.tsv"
OUT_FILE_3 = "task3.tsv"
SUBMISSION_ZIP = f"submission_{PHASE}.zip"

# ─────────────────────────────────────────
# EMBEDDINGS
# ─────────────────────────────────────────
EMBED_MODEL = "intfloat/multilingual-e5-large-instruct" #"BAAI/bge-m3" | "intfloat/multilingual-e5-large"| "intfloat/multilingual-e5-large-instruct"(0.93, 0.75, 1.0)

# ─────────────────────────────────────────
# OLLAMA — LLM
# ─────────────────────────────────────────
OLLAMA_MODE = os.environ.get("OLLAMA_MODE", "cloud")  # "local" | "cloud"

if OLLAMA_MODE == "cloud":
    OLLAMA_HOST = "https://ollama.com"
    OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY", "")
    LLM_MODEL = os.environ.get("LLM_MODEL", "glm-5.1:cloud") #"glm-5.1:cloud" gemma4:31b-cloud
else:
    OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
    OLLAMA_API_KEY = None
    LLM_MODEL = os.environ.get("LLM_MODEL", "llama3.2")

# Pesos del ensemble
ENSEMBLE_WEIGHT_EMB = 0.4
ENSEMBLE_WEIGHT_LLM = 0.6
USE_LLM_FALLBACK = True

print(f"PHASE       : {PHASE}")
print(f"OLLAMA_MODE : {OLLAMA_MODE}")
print(f"OLLAMA_HOST : {OLLAMA_HOST}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print(f"EMBED_MODEL : {EMBED_MODEL}")
print()
print(f"EVAL T1: {EVAL_FILE_1} (existe: {Path(EVAL_FILE_1).exists()})")
print(f"EVAL T2: {EVAL_FILE_2} (existe: {Path(EVAL_FILE_2).exists()})")
print(f"EVAL T3: {EVAL_FILE_3} (existe: {Path(EVAL_FILE_3).exists()})")

PHASE       : test
OLLAMA_MODE : cloud
OLLAMA_HOST : https://ollama.com
LLM_MODEL   : glm-5.1:cloud
EMBED_MODEL : intfloat/multilingual-e5-large-instruct

EVAL T1: haha2026_test/task1_test.tsv (existe: True)
EVAL T2: haha2026_test/task2_test.tsv (existe: True)
EVAL T3: haha2026_test/task3_test.tsv (existe: True)


## 3. Imports y utilidades base

In [ ]:
import pandas as pd
import numpy as np
import re
import time
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer


def load_tsv(path: str) -> pd.DataFrame:
    if not Path(path).exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t")


def save_predictions(df: pd.DataFrame, preds: list, path: str, col: str = "tag"):
    out = df[["id"]].copy()
    out[col] = preds
    out.to_csv(path, sep="\t", index=False)
    print(f"  Guardado: {path} ({len(out)} filas)")


def merge_labeled(df_trial: pd.DataFrame, df_dev: pd.DataFrame) -> pd.DataFrame:
    """Combina trial + dev SOLO si dev tiene la columna 'tag' (etiquetado).
    De lo contrario devuelve solo trial."""
    if df_dev is None or df_dev.empty or "tag" not in df_dev.columns:
        return df_trial.copy()
    common_cols = [c for c in df_trial.columns if c in df_dev.columns]
    return pd.concat(
        [df_trial[common_cols], df_dev[common_cols]], ignore_index=True
    )

## 4. Modelo de embeddings

In [ ]:
print(f"Cargando modelo de embeddings: {EMBED_MODEL}")
encoder = SentenceTransformer(EMBED_MODEL)


def embed(texts):
    return encoder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )


def cosine_sim(a, b):
    return a @ b.T

Cargando modelo de embeddings: intfloat/multilingual-e5-large-instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

## 5. Cliente LLM (Ollama)

In [ ]:
import ollama

if OLLAMA_MODE == "cloud":
    llm_client = ollama.Client(
        host=OLLAMA_HOST,
        headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"} if OLLAMA_API_KEY else None,
    )
else:
    llm_client = ollama.Client(host=OLLAMA_HOST)


def llm_generate(prompt: str, system: str = None, temperature: float = 0.3, max_retries: int = 2) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    for attempt in range(max_retries):
        try:
            resp = llm_client.chat(
                model=LLM_MODEL,
                messages=messages,
                options={"temperature": temperature},
            )
            return resp["message"]["content"].strip()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  ⚠️  LLM error: {e}")
                return ""
            time.sleep(1)
    return ""


try:
    test_response = llm_generate("Responde solo con la palabra OK.")
    print(f"✓ LLM disponible. Test: {test_response[:50]}")
    LLM_AVAILABLE = bool(test_response)
except Exception as e:
    print(f"⚠️  LLM no disponible: {e}")
    LLM_AVAILABLE = False

✓ LLM disponible. Test: OK


## 6. Clasificador de embeddings

In [ ]:
class EmbeddingClassifier:
    """KNN ponderado o LogReg sobre embeddings, con probabilidades."""

    def __init__(self, strategy="knn", k=5):
        self.strategy = strategy
        self.k = k
        self.X_train = None
        self.y_train = None
        self.classes_ = None
        self.clf = LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        )

    def fit(self, X, y):
        self.X_train = X
        self.y_train = np.array(y)
        self.classes_ = sorted(set(y))
        if self.strategy == "logreg" and len(self.classes_) >= 2:
            self.clf.fit(X, y)

    def predict_proba(self, X):
        if self.strategy == "logreg" and hasattr(self.clf, "classes_"):
            probs = self.clf.predict_proba(X)
            return [
                {c: float(p) for c, p in zip(self.clf.classes_, row)}
                for row in probs
            ]
        results = []
        for vec in X:
            sims = cosine_sim(vec, self.X_train)
            top_k = np.argsort(sims)[::-1][: self.k]
            labels = self.y_train[top_k]
            weights = sims[top_k]
            scores = {c: 0.0 for c in self.classes_}
            for lbl, w in zip(labels, weights):
                scores[lbl] += float(w)
            total = sum(scores.values()) or 1.0
            results.append({c: s / total for c, s in scores.items()})
        return results

    def predict(self, X):
        return [max(p, key=p.get) for p in self.predict_proba(X)]

## 7. LLM como clasificador

In [ ]:
TASK1_SYSTEM = (
    "Eres un clasificador de noticias en español. Determinas si un titular "
    "corresponde a una noticia real o a una noticia satírica/humorística. "
    "Responde SOLO con una palabra: 'satirical' o 'real'."
)

TASK2_SYSTEM = (
    "Eres un detector de texto generado por IA. Dado un titular y un chiste "
    "asociado, determinas si el chiste fue escrito por un humano o generado "
    "por una máquina (LLM). Responde SOLO con una palabra: 'machine' o 'human'."
)


def parse_label(response: str, valid_labels: list):
    if not response:
        return None
    text = response.lower().strip()
    for lbl in valid_labels:
        if lbl.lower() in text:
            return lbl
    return None


def llm_classify_task1(headline: str, context: str = "") -> dict:
    prompt = f"Titular: {headline}\n"
    if context and str(context) != "nan":
        prompt += f"Contexto: {str(context)[:300]}\n"
    prompt += "\n¿Es 'satirical' o 'real'? Responde con una sola palabra."
    resp = llm_generate(prompt, system=TASK1_SYSTEM, temperature=0.0)
    label = parse_label(resp, ["satirical", "real"])
    if label == "satirical":
        return {"satirical": 0.85, "real": 0.15}
    if label == "real":
        return {"satirical": 0.15, "real": 0.85}
    return {"satirical": 0.5, "real": 0.5}


def llm_classify_task2(headline: str, joke: str) -> dict:
    prompt = (
        f"Titular: {headline}\n"
        f"Chiste: {joke}\n\n"
        "¿El chiste fue 'machine' (generado por IA) o 'human' (escrito por humano)? "
        "Pistas: el texto generado por IA tiende a ser más estructurado, predecible, "
        "con frases hechas y poca espontaneidad. El humano suele ser más coloquial, "
        "breve, con regionalismos o referencias locales. Responde con una sola palabra."
    )
    resp = llm_generate(prompt, system=TASK2_SYSTEM, temperature=0.0)
    label = parse_label(resp, ["machine", "human"])
    if label == "machine":
        return {"machine": 0.85, "human": 0.15}
    if label == "human":
        return {"machine": 0.15, "human": 0.85}
    return {"machine": 0.5, "human": 0.5}

## 8. Combinador del ensemble

In [ ]:
def ensemble_predict(emb_probs, llm_probs, weight_emb=ENSEMBLE_WEIGHT_EMB, weight_llm=ENSEMBLE_WEIGHT_LLM):
    if llm_probs is None:
        return [max(p, key=p.get) for p in emb_probs]
    final = []
    for ep, lp in zip(emb_probs, llm_probs):
        labels = set(ep) | set(lp)
        combined = {
            l: weight_emb * ep.get(l, 0.0) + weight_llm * lp.get(l, 0.0)
            for l in labels
        }
        final.append(max(combined, key=combined.get))
    return final

## 9. Task 1 — Humor Detection (test)

In [ ]:
def build_text_t1(row):
    headline = str(row.get("headline", "")).strip()
    ctx = str(row.get("context", "")).strip()[:200]
    ctx = "" if ctx == "nan" else ctx
    return f"{headline} {ctx}".strip()


def run_task1_ensemble(df_train, df_eval):
    print("\n" + "━" * 55)
    print("TASK 1: Humor Detection (Ensemble) — fase: " + PHASE)
    print("━" * 55)

    train_texts = [build_text_t1(r) for _, r in df_train.iterrows()]
    train_labels = df_train["tag"].tolist()
    eval_texts = [build_text_t1(r) for _, r in df_eval.iterrows()]

    print(f"  Train: {len(train_texts)} | Eval: {len(eval_texts)}")
    print("  [A] Embeddings train...")
    X_train = embed(train_texts)
    print("  [A] Embeddings eval...")
    X_eval = embed(eval_texts)

    strategy = "logreg" if len(train_texts) >= 20 else "knn"
    print(f"  [A] Estrategia: {strategy}")
    clf = EmbeddingClassifier(strategy=strategy, k=5)
    clf.fit(X_train, train_labels)
    emb_probs = clf.predict_proba(X_eval)

    llm_probs = None
    if LLM_AVAILABLE:
        print("  [B] Clasificación LLM...")
        llm_probs = []
        for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="LLM T1"):
            llm_probs.append(
                llm_classify_task1(
                    str(row.get("headline", "")), str(row.get("context", ""))
                )
            )
    elif not USE_LLM_FALLBACK:
        raise RuntimeError("LLM no disponible y fallback desactivado")
    else:
        print("  [B] LLM no disponible — usando solo embeddings")

    return ensemble_predict(emb_probs, llm_probs)

## 10. Task 2 — LLM-generated detection (test)

In [ ]:
def build_text_t2(row, joke_col):
    headline = str(row.get("headline", "")).strip()
    joke = str(row.get(joke_col, "")).strip()
    joke = "" if joke == "nan" else joke
    return f"{headline} || {joke}".strip()


def run_task2_ensemble(df_train, df_eval, joke_col="joke"):
    print("\n" + "━" * 55)
    print("TASK 2: LLM-generated Humor Detection (Ensemble) — fase: " + PHASE)
    print("━" * 55)

    if joke_col not in df_train.columns:
        joke_col = "headline"

    train_texts = [build_text_t2(r, joke_col) for _, r in df_train.iterrows()]
    train_labels = df_train["tag"].tolist()
    eval_texts = [build_text_t2(r, joke_col) for _, r in df_eval.iterrows()]

    print(f"  Train: {len(train_texts)} | Eval: {len(eval_texts)}")
    print("  [A] Embeddings train...")
    X_train = embed(train_texts)
    print("  [A] Embeddings eval...")
    X_eval = embed(eval_texts)

    strategy = "logreg" if len(train_texts) >= 20 else "knn"
    print(f"  [A] Estrategia: {strategy}")
    clf = EmbeddingClassifier(strategy=strategy, k=5)
    clf.fit(X_train, train_labels)
    emb_probs = clf.predict_proba(X_eval)

    llm_probs = None
    if LLM_AVAILABLE:
        print("  [B] Clasificación LLM...")
        llm_probs = []
        for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="LLM T2"):
            llm_probs.append(
                llm_classify_task2(
                    str(row.get("headline", "")), str(row.get(joke_col, ""))
                )
            )
    elif not USE_LLM_FALLBACK:
        raise RuntimeError("LLM no disponible y fallback desactivado")
    else:
        print("  [B] LLM no disponible — usando solo embeddings")

    return ensemble_predict(emb_probs, llm_probs)

## 11. Task 3 — Humor Generation (RAG + LLM, test)

In [ ]:
TASK3_SYSTEM = (
    "Eres un comediante experto en escribir chistes en español a partir de "
    "titulares de noticias. Tus chistes son concisos, creativos y genuinamente "
    "divertidos. Solo devuelves el chiste, sin explicación ni introducción."
)

BASELINE_PROMPT = (
    'Create a joke based on this title of a news article:\n\n'
    '"{headline}"\n\n'
    "The joke should be concise, creative and genuinely funny. "
    "Only return the joke and nothing else. All jokes must be in spanish."
)


def build_fewshot_block(retrieved):
    if not retrieved:
        return ""
    lines = ["Aquí tienes ejemplos de buenos chistes humanos sobre titulares (úsalos solo como referencia de estilo):\n"]
    for h, j in retrieved:
        lines.append(f"Titular: {h}\nChiste: {j}\n")
    lines.append("Ahora genera un chiste nuevo, original, en español, conciso y gracioso para el siguiente titular.\n")
    return "\n".join(lines)


def clean_joke(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    for prefix in ["Chiste:", "chiste:", "Joke:", "Aquí tienes", "Aquí está"]:
        if t.lower().startswith(prefix.lower()):
            t = t[len(prefix):].strip()
    if (t.startswith('"') and t.endswith('"')) or (t.startswith("'") and t.endswith("'")):
        t = t[1:-1].strip()
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = t.replace("\t", " ")
    return t.strip()


def build_rag_pool(df_trial_2, df_dev_2):
    """Pool de ejemplos humanos para RAG: trial T2 + (dev T2 si tiene tag)."""
    frames = []
    if df_trial_2 is not None and not df_trial_2.empty and "joke" in df_trial_2.columns:
        df = df_trial_2.copy()
        if "tag" in df.columns:
            df = df[df["tag"] == "human"]
        frames.append(df[["headline", "joke"]])
    if df_dev_2 is not None and not df_dev_2.empty and "joke" in df_dev_2.columns and "tag" in df_dev_2.columns:
        df = df_dev_2[df_dev_2["tag"] == "human"]
        frames.append(df[["headline", "joke"]])
    if not frames:
        return pd.DataFrame(columns=["headline", "joke"])
    pool = pd.concat(frames, ignore_index=True)
    pool = pool[pool["joke"].notna() & (pool["joke"].astype(str) != "nan")]
    return pool.drop_duplicates(subset=["headline", "joke"]).reset_index(drop=True)


def run_task3_rag_llm(rag_pool, df_eval, k_fewshot=3):
    print("\n" + "━" * 55)
    print("TASK 3: Humor Generation (RAG + LLM) — fase: " + PHASE)
    print("━" * 55)

    if not LLM_AVAILABLE:
        print("  ⚠️  LLM no disponible — fallback a placeholder")
        return [
            "Otra noticia más para sumar a la lista de cosas inexplicables."
            for _ in df_eval.iterrows()
        ]

    use_rag = rag_pool is not None and not rag_pool.empty
    rag_headlines, rag_jokes, X_rag = [], [], None

    if use_rag:
        rag_headlines = rag_pool["headline"].astype(str).tolist()
        rag_jokes = rag_pool["joke"].astype(str).tolist()
        print(f"  [RAG] Indexando {len(rag_headlines)} ejemplos humanos...")
        X_rag = embed(rag_headlines)
    else:
        print("  [RAG] Sin ejemplos — generación pura")

    eval_headlines = df_eval["headline"].astype(str).tolist()
    if use_rag:
        print("  [RAG] Embeddings eval...")
        X_eval = embed(eval_headlines)

    print("  [LLM] Generando chistes...")
    jokes = []
    for i, headline in enumerate(tqdm(eval_headlines, desc="LLM T3")):
        retrieved = []
        if use_rag:
            sims = cosine_sim(X_eval[i], X_rag)
            top_k = np.argsort(sims)[::-1][:k_fewshot]
            retrieved = [(rag_headlines[idx], rag_jokes[idx]) for idx in top_k]

        prompt = build_fewshot_block(retrieved) + BASELINE_PROMPT.format(headline=headline)
        response = llm_generate(prompt, system=TASK3_SYSTEM, temperature=0.8)
        joke = clean_joke(response)

        if not joke:
            joke = f"Justo cuando creías que '{headline[:40]}...' no podía sorprenderte más."
        jokes.append(joke)

    return jokes

## 12. Pipeline principal — fase TEST

In [ ]:
def main():
    # ── Cargar trial (etiquetado) ──
    df_trial_1 = load_tsv(TRIAL_FILE_1)
    df_trial_2 = load_tsv(TRIAL_FILE_2)

    # ── Cargar dev (puede o no tener tag) ──
    df_dev_1 = load_tsv(DEV_FILE_1)
    df_dev_2 = load_tsv(DEV_FILE_2)
    df_dev_3 = load_tsv(DEV_FILE_3)

    # ── Cargar archivos de evaluación (test o dev según PHASE) ──
    df_eval_1 = load_tsv(EVAL_FILE_1)
    df_eval_2 = load_tsv(EVAL_FILE_2)
    df_eval_3 = load_tsv(EVAL_FILE_3)

    print(f"Trial T1: {len(df_trial_1)} | Dev T1: {len(df_dev_1)} | Eval T1: {len(df_eval_1)}")
    print(f"Trial T2: {len(df_trial_2)} | Dev T2: {len(df_dev_2)} | Eval T2: {len(df_eval_2)}")
    print(f"Dev T3: {len(df_dev_3)} | Eval T3: {len(df_eval_3)}")

    # ── Conjunto de entrenamiento: trial + dev (si dev tiene tag) ──
    df_train_1 = merge_labeled(df_trial_1, df_dev_1)
    df_train_2 = merge_labeled(df_trial_2, df_dev_2)
    print(f"\nTrain T1 final: {len(df_train_1)} | Train T2 final: {len(df_train_2)}")

    # ── TASK 1 ──
    if not df_eval_1.empty and not df_train_1.empty and "tag" in df_train_1.columns:
        preds_t1 = run_task1_ensemble(df_train_1, df_eval_1)
        save_predictions(df_eval_1, preds_t1, OUT_FILE_1)
    else:
        print("\n⚠️  Task 1 omitido (faltan datos)")

    # ── TASK 2 ──
    if not df_eval_2.empty and not df_train_2.empty and "tag" in df_train_2.columns:
        joke_col = "joke" if "joke" in df_eval_2.columns else "headline"
        preds_t2 = run_task2_ensemble(df_train_2, df_eval_2, joke_col=joke_col)
        save_predictions(df_eval_2, preds_t2, OUT_FILE_2)
    else:
        print("\n⚠️  Task 2 omitido (faltan datos)")

    # ── TASK 3 ──
    if not df_eval_3.empty:
        rag_pool = build_rag_pool(df_trial_2, df_dev_2)
        jokes = run_task3_rag_llm(rag_pool, df_eval_3, k_fewshot=3)
        save_predictions(df_eval_3, jokes, OUT_FILE_3, col="text")
        print("\n  Preview T3:")
        for i, (_, row) in enumerate(df_eval_3.head(5).iterrows()):
            print(f"\n  Titular: {row['headline'][:80]}")
            print(f"  Chiste : {jokes[i][:120]}")
    else:
        print("\n⚠️  Task 3 omitido (eval vacío)")

    # ── ZIP de submission ──
    import zipfile
    files = [f for f in [OUT_FILE_1, OUT_FILE_2, OUT_FILE_3] if Path(f).exists()]
    if files:
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f)
        print(f"\n  Submission: {SUBMISSION_ZIP} ({', '.join(files)})")

    print("\n✓ Listo.")


main()

Trial T1: 24 | Dev T1: 540 | Eval T1: 600
Trial T2: 12 | Dev T2: 350 | Eval T2: 550
Dev T3: 0 | Eval T3: 300

Train T1 final: 564 | Train T2 final: 362

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 1: Humor Detection (Ensemble) — fase: test
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train: 564 | Eval: 600
  [A] Embeddings train...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

  [A] Embeddings eval...


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

  [A] Estrategia: logreg
  [B] Clasificación LLM...


...

  [RAG] Embeddings eval...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

  [LLM] Generando chistes...


LLM T3: 100%|██████████| 300/300 [2:00:56<00:00, 24.19s/it]

  Guardado: task3.tsv (300 filas)

  Preview T3:

  Titular: Final del Mundial 2026 contará con show musical de Shakira, Madonna y BTS
  Chiste : Los jugadores van a hacer playback.

  Titular: Imagen de un alcatraz cubierta de pasto gana el Premio del Público en los Comedy
  Chiste : El alcatraz aceptó el premio, pero aclaró: "No es pasto, es un look ecológico."

  Titular: 6 unidades de metrobus se unen al plan de contigencia por lluvias del Metro de L
  Chiste : 6 autobuses: el plan de contingencia oficial para que también nos mojemos en la superficie.

  Titular: Diferencias entre mareo y vértigo: qué síntomas indican que debes acudir a urgen
  Chiste : Mareo es cuando ves doble, vértigo es cuando ves la cuenta de urgencias.

  Titular: Tecnología, inversión y futuro: los ejes del segundo Foro de Logística
  Chiste : Es el segundo porque al primero lo devolvieron por dirección incorrecta.

  Submission: submission_test.zip (task1.tsv, task2.tsv, task3.tsv)

✓ Listo.


## 13. (Opcional) Verificación final del formato

Comprueba que los TSV generados respetan el formato pedido en HAHA 2026.

In [ ]:
def check_submission_file(path, expected_col, eval_path):
    if not Path(path).exists():
        print(f"  {path}: NO existe")
        return
    sub = pd.read_csv(path, sep="\t")
    ev = pd.read_csv(eval_path, sep="\t")
    ok_cols = list(sub.columns) == ["id", expected_col]
    ok_count = len(sub) == len(ev)
    ok_order = sub["id"].tolist() == ev["id"].tolist()
    status = "✓" if (ok_cols and ok_count and ok_order) else "✗"
    print(f"  {status} {path}: cols={list(sub.columns)} | n={len(sub)}/{len(ev)} | orden={'OK' if ok_order else 'NO'}")


if Path(EVAL_FILE_1).exists():
    check_submission_file(OUT_FILE_1, "tag", EVAL_FILE_1)
if Path(EVAL_FILE_2).exists():
    check_submission_file(OUT_FILE_2, "tag", EVAL_FILE_2)
if Path(EVAL_FILE_3).exists():
    check_submission_file(OUT_FILE_3, "text", EVAL_FILE_3)

  ✓ task1.tsv: cols=['id', 'tag'] | n=600/600 | orden=OK
  ✓ task2.tsv: cols=['id', 'tag'] | n=550/550 | orden=OK
  ✓ task3.tsv: cols=['id', 'text'] | n=300/300 | orden=OK
